In [1]:
import pandas as pd

In [4]:
huri_df = pd.read_csv("../data/interactomes/HuRI/HuRI.tsv", sep="\t", header=None)
huri_df.head()

,0,1
0,ENSG00000000005,ENSG00000061656
1,ENSG00000000005,ENSG00000099968
2,ENSG00000000005,ENSG00000104765
3,ENSG00000000005,ENSG00000105383
4,ENSG00000000005,ENSG00000114455


In [ ]:
gene_space_df = pd.read_csv("../data/interactomes/HuRI/Supplementary Table 1.txt", sep="\t")

In [ ]:
gene_space_df.head()

In [ ]:
transcripts_df = pd.read_csv("../data/interactomes/HuRI/Supplementary Table 2.txt", sep="\t")


,orf_id,ensembl_transcript_id,ensembl_protein_id,ensembl_gene_id,symbol
0,1,ENST00000272298.11,ENSP00000272298.7,ENSG00000143933.16,CALM2
1,1,ENST00000356978.8,ENSP00000349467.4,ENSG00000198668.10,CALM1
2,1,ENST00000291295.13,ENSP00000291295.8,ENSG00000160014.16,CALM3
3,2,ENST00000327247.9,ENSP00000314508.5,ENSG00000177628.15,GBA
4,3,ENST00000222547.7,ENSP00000222547.3,ENSG00000105829.11,BET1


In [ ]:
transcripts_df.head()

In [ ]:
transcripts_df = (
    transcripts_df.assign(gene_id_unversioned=transcripts_df["ensembl_gene_id"].str.split(".").str[0])
    .merge(gene_space_df, left_on="gene_id_unversioned", right_on="Ensembl_gene_id", how="left")
    .drop(columns=["gene_id_unversioned", "Ensembl_gene_id"])
)


,orf_id,ensembl_transcript_id,ensembl_protein_id,ensembl_gene_id,symbol,in_space_3,in_GTEx,in_HPA,in_FANTOM
0,1,ENST00000272298.11,ENSP00000272298.7,ENSG00000143933.16,CALM2,1,1,1,1
1,1,ENST00000356978.8,ENSP00000349467.4,ENSG00000198668.10,CALM1,1,1,1,1
2,1,ENST00000291295.13,ENSP00000291295.8,ENSG00000160014.16,CALM3,1,1,1,1
3,2,ENST00000327247.9,ENSP00000314508.5,ENSG00000177628.15,GBA,1,1,1,1
4,3,ENST00000222547.7,ENSP00000222547.3,ENSG00000105829.11,BET1,1,1,1,1


In [ ]:
transcripts_df.head()

In [9]:
print(f"Unmatched rows: {(n := transcripts_df['in_space_3'].isna().sum())}")
assert n == 0, f"{n} unmatched rows"

Unmatched rows: 0


In [10]:
print(f"Search space size: {len(gene_space_df)}")

Search space size: 19818


In [11]:
n_space_3 = gene_space_df["in_space_3"].sum()
print(f"Space 3 size: {n_space_3} ({n_space_3 / len(gene_space_df):.2%} of search space)")

Space 3 size: 17408 (87.84% of search space)


In [12]:
huri_genes = set(huri_df[0]) | set(huri_df[1])
transcript_genes = set(transcripts_df["ensembl_gene_id"].str.split(".").str[0])
n_in = len(huri_genes & transcript_genes)
print(f"HuRI genes in transcripts_df: {n_in} ({n_in / len(transcripts_df):.2%} of transcripts_df rows)")

HuRI genes in transcripts_df: 8272 (47.44% of transcripts_df rows)


In [13]:
n_out = len(huri_genes - transcript_genes)
print(f"HuRI genes NOT in transcripts_df: {n_out} ({n_out / len(transcripts_df):.2%} of transcripts_df rows)")

HuRI genes NOT in transcripts_df: 0 (0.00% of transcripts_df rows)


In [14]:
print(f"Rows without protein ID: {(n := transcripts_df['ensembl_protein_id'].isna().sum())}")
assert n == 0, f"{n} rows without protein ID"

Rows without protein ID: 0


In [15]:
protein_ids = transcripts_df["ensembl_protein_id"].dropna().drop_duplicates()
protein_ids.to_csv("../data/interactomes/HuRI/proteins.txt", index=False, header=False)
print(f"Wrote {len(protein_ids)} protein IDs")

Wrote 17421 protein IDs
